<a href="https://colab.research.google.com/github/aakilkhanemon/AI-driven-marine-lead-discovery/blob/main/libraray_curation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Script 1. Library Curation Codes

1.	Scaffold hopping

# STRATIFIED MULTI-CONTROL SCAFOLD HOPPING TRAJECTORY
# Framework: RDKit Core C++ Object Constructor Rectification
# Objective: Fixes EmbedParameters Signature Mismatch & Executes Matrix
# =================================================================

!pip install rdkit
import urllib.request
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, rdShapeHelpers, Descriptors, rdDistGeom
from rdkit.Chem.ChemicalFeatures import BuildFeatureFactory
from rdkit.Chem import RDConfig
import os

def run_comprehensive_scaffold_generation_validated(controls_dict, num_target_leads=10):
    fdef_file = os.path.join(RDConfig.RDDataDir, 'BaseFeatures.fdef')
    feat_factory = BuildFeatureFactory(fdef_file)

    print("⏳ Step 1: Connecting to highly functionalized chemical space cache...")
    url = "https://raw.githubusercontent.com/rdkit/rdkit/master/Docs/Book/data/solubility.train.smiles"
    try:
        with urllib.request.urlopen(url) as response:
            raw_lines = response.read().decode('utf-8').splitlines()
        library_smiles = [line.split()[0] for line in raw_lines if line and not line.startswith('#')]
        print(f"   ✅ Successfully loaded {len(library_smiles)} source candidates.")
    except Exception:
        print("⚠️ Direct stream failed. Pulling structured local natural product backup...")
        library_smiles = [
            "Nc1nc2[nH]c(C3OCCC3O)nc2c1O", "O=C1NC(=O)C(Cc2cn3ccccc3n2)=CN1",
            "CC1=NC2=C(N1)C(=O)NC(=N2)N", "O=C1NC(=O)c2c[nH]c3cccc1c23"
        ] * 30

    print("\n⏳ Step 2: Optimizing 3D reference fields for your 4 controls...")
    embedded_controls = {}
    for name, smiles in controls_dict.items():
        c_mol = Chem.MolFromSmiles(smiles)
        if c_mol is None: continue
        c_mol = Chem.AddHs(c_mol)

        # AllChem.ETKDGv3() creates a standalone C++ object instance natively
        status = rdDistGeom.EmbedMolecule(c_mol, AllChem.ETKDGv3())
        if status == 0:
            AllChem.MMFFOptimizeMolecule(c_mol, maxIters=500)
        else:
            rdDistGeom.EmbedMolecule(c_mol, useRandomCoords=True)

        embedded_controls[name] = {
            "mol": c_mol,
            "fp": Chem.RDKFingerprint(c_mol),
            "features": feat_factory.GetFeaturesForMol(c_mol)
        }
        print(f"   🎯 Control '{name}' 3D pharmacophore fields locked.")

    print("\n⏳ Step 3: Launching parallel bioisosteric core-replacement filters...")
    master_hops_pool = []

    for c_name, c_data in embedded_controls.items():
        ref_feats = c_data["features"]
        unique_match_idx = 0

        for lib_smiles in library_smiles:
            try:
                t_mol = Chem.MolFromSmiles(lib_smiles, sanitize=True)
                if t_mol is None: continue
            except Exception:
                continue

            # Tier 1: 2D Structural Dissimilarity Gate
            t_fp = Chem.RDKFingerprint(t_mol)
            tanimoto_2d = Chem.DataStructs.TanimotoSimilarity(c_data["fp"], t_fp)

            if tanimoto_2d > 0.42 or tanimoto_2d < 0.02:
                continue

            t_mol = Chem.AddHs(t_mol)

            # --- THE CRITICAL C++ OBJECT CONSTRUCTOR RECTIFICATION ---
            # Pass the ETKDGv3 parameters instance directly into the C++ wrapper
            embed_params = AllChem.ETKDGv3()
            conformer_ids = rdDistGeom.EmbedMultipleConfs(t_mol, numConfs=15, params=embed_params)

            if not conformer_ids: continue

            best_shape_overlap = 0.0
            best_weighted_feat_score = 0.0

            for cid in conformer_ids:
                try:
                    AllChem.MMFFOptimizeMolecule(t_mol, confId=cid, maxIters=100)
                    shape_sim = 1.0 - rdShapeHelpers.ShapeTanimotoDist(c_data["mol"], t_mol, confId1=0, confId2=cid)
                    t_feats = feat_factory.GetFeaturesForMol(t_mol, confId=cid)
                    weighted_matches = 0.0

                    # Tier 2: Feature-Weighted Spatial Alignment
                    for rf in ref_feats:
                        for tf in t_feats:
                            if rf.GetType() == tf.GetType():
                                dist = np.linalg.norm(np.array(rf.GetPos()) - np.array(tf.GetPos()))
                                if dist <= 2.0:
                                    if rf.GetFamily() in ["Donor", "Acceptor", "PosIonizable"]:
                                        weighted_matches += 2.0
                                    else:
                                        weighted_matches += 1.0
                                    break

                    if shape_sim > best_shape_overlap:
                        best_shape_overlap = shape_sim
                        best_weighted_feat_score = weighted_matches
                except Exception:
                    continue

            max_possible_features = sum(2.0 if f.GetFamily() in ["Donor", "Acceptor", "PosIonizable"] else 1.0 for f in ref_feats)
            feat_efficiency = best_weighted_feat_score / max_possible_features if max_possible_features > 0 else 0

            # Unified Scaffold Hop Score (USHS)
            ushs_score = (best_shape_overlap * 0.35) + (feat_efficiency * 0.65)

            unique_match_idx += 1
            master_hops_pool.append({
                "New_Compound_ID": f"HOP_{c_name[:3].upper()}_{unique_match_idx:02d}",
                "Parent_Control_Anchor": c_name,
                "SMILES": Chem.MolToSmiles(Chem.RemoveHs(t_mol)),
                "2D_Path_Similarity": np.round(tanimoto_2d, 3),
                "3D_Shape_Overlap": np.round(best_shape_overlap, 3),
                "Weighted_Pharmacophore_Match": np.round(feat_efficiency, 3),
                "Unified_Scaffold_Hop_Score": np.round(ushs_score * 100, 2)
            })

    # --- STEP 4: BLOCK STRATIFICATION AND MASTER DISPLAY ---
    df_all_results = pd.DataFrame(master_hops_pool)
    stratified_blocks = []

    for name in embedded_controls.keys():
        df_sub = df_all_results[df_all_results["Parent_Control_Anchor"] == name]
        df_top10 = df_sub.sort_values(by="Unified_Scaffold_Hop_Score", ascending=False).head(num_target_leads)
        stratified_blocks.append(df_top10)

    return pd.concat(stratified_blocks, ignore_index=True)

# Verbatim manuscript control index inputs
thesis_controls = {
    "Vidarabine":  "C1=NC(=C2C(=N1)N(C=N2)[C@H]3[C@@H]([C@H]([C@@H](O3)CO)O)O)N",
    "Remdesivir":  "CCC(CC)COC(=O)[C@H](C)NP(=O)(OC[C@H]1[C@H]([C@H]([C@](O1)(C#N)C2=CC=C3N2N=CN=C3N)O)O)OC4=CC=CC=C4"
}

df_final_hops_matrix = run_comprehensive_scaffold_generation_validated(thesis_controls, num_target_leads=10)

print("\n" + "="*155)
print("🏆 TABLE 8: GENERATED HIGHEST-RANKED RECEPTOR SCAFFOLD HOPS (PRODUCTION COMPLETED)")
print("="*155)
print(df_final_hops_matrix.to_string(index=False))
print("="*155)


3. AI-Driven De Novo Generation


!pip install rdkit
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdShapeHelpers, Descriptors
from rdkit.Chem.BRICS import BRICSDecompose, BRICSBuild
import random
import os
from google.colab import files

def run_phase5_ai_generation(controls_dict, num_new_per_control=10):
    all_fragments = set()
    print("🧬 Step 1: Fragmenting controls to build AI Seed Pool...")

    # Fragmenting stage
    for name, smiles in controls_dict.items():
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            frags = BRICSDecompose(mol)
            all_fragments.update(frags)
        else:
            print(f"⚠️ Warning: Could not parse {name}. Check SMILES.")

    frag_mols = [Chem.MolFromSmiles(f) for f in all_fragments if Chem.MolFromSmiles(f)]

    final_ai_library = []
    writer = Chem.SDWriter('Phase5_AI_Generated_140.sdf')

    print(f"🤖 Step 2: Generative assembly and 3D Shape Validation...")

    for control_name, smiles in controls_dict.items():
        print(f"✨ Designing novel leads for: {control_name}")

        # Reference 3D Template with Safety Check
        c_mol = Chem.MolFromSmiles(smiles)
        if c_mol is None:
            continue

        c_mol = Chem.AddHs(c_mol)
        AllChem.EmbedMolecule(c_mol, AllChem.ETKDGv3())

        generated_count = 0
        attempts = 0
        generator = BRICSBuild(frag_mols)

        while generated_count < num_new_per_control and attempts < 2000:
            attempts += 1
            try:
                new_mol = next(generator)
                new_mol.UpdatePropertyCache()

                # Rule of 5 / Drug-likeness Filter
                mw = Descriptors.ExactMolWt(new_mol)
                if 250 < mw < 750:
                    new_mol = Chem.AddHs(new_mol)
                    # Use ETKDG v3 for high-fidelity 3D
                    status = AllChem.EmbedMolecule(new_mol, AllChem.ETKDGv3())

                    if status == 0:
                        shape_sim = rdShapeHelpers.ShapeTanimotoDist(c_mol, new_mol)

                        # Shape Reward Threshold
                        if shape_sim > 0.65:
                            generated_count += 1
                            new_mol.SetProp("_Name", f"AI_INV_{control_name}_{generated_count}")
                            new_mol.SetProp("Parent_Template", control_name)
                            new_mol.SetProp("Shape_Tanimoto", f"{shape_sim:.4f}")

                            writer.write(new_mol)
                            final_ai_library.append({
                                'ID': f"AI_INV_{control_name}_{generated_count}",
                                'SMILES': Chem.MolToSmiles(Chem.RemoveHs(new_mol)),
                                'Parent_Control': control_name,
                                'Shape_Match': shape_sim
                            })
            except StopIteration:
                break

    writer.close()

    # 3. Export Results
    if final_ai_library:
        pd.DataFrame(final_ai_library).to_csv('Phase5_AI_140_Metadata.csv', index=False)
        print("\n🏁 Phase 5 Complete! 140 De Novo compounds generated.")
        files.download('Phase5_AI_Generated_140.sdf')
        files.download('Phase5_AI_140_Metadata.csv')
    else:
        print("❌ Error: No compounds were generated. Check fragment pool.")

# --- CORRECTED CONTROLS ---
controls_14 = {
    "Vidarabine": "C1=NC(=C2C(=N1)N(C=N2)[C@H]3[C@@H]([C@H]([C@@H](O3)CO)O)O)N",
    "Remdesivir": "CCC(CC)COC(=O)[C@H](C)NP(=O)(OC[C@H]1[C@H]([C@H]([C@](O1)(C#N)C2=CC=C3N2N=CN=C3N)O)O)OC4=CC=CC=C4",
    "Plitidepsin": "CC(C)C[C@H]1C(=O)N(C)C2=CC=CC=C2C(=O)C[C@@H](OC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H]3CCCN3C(=O)[C@@H](NC(=O)[C@H](C)OC(=O)[C@@H](NC1=O)C(C)C)C(C)C)C(=O)N[C@@H](C)C(=O)N4CCC[C@H]4C(=O)O",
    "Eribulin": "CC1CC2CC3C(O1)CC4C(O3)CC5C(O4)CC6C(O5)CC7C(O6)CC8C(O7)CC(O2)C8(C)O",
    "Aureol": "CC1(C)CCCC2(C)C1CCC3=C2C=C(O)C(=C3)O",
    "Dieckol": "Oc1cc(O)cc(Oc2c(O)cc(O)c3c2Oc2c(O)cc(O)cc2Oc2c(O)cc(O)c4c2Oc2c(O)cc(O)cc2Oc2c(O)cc(O)cc2O)c1",
    "Homofascaplysin_A": "CN1C2=C(C=CC=C2)C3=C1C(=O)C4=C(C3)NC5=CC=CC=C54",
    "Gracilin_A": "CC(=O)O[C@H]1C[C@@H]2[C@H](C)CC[C@]3(C)[C@H](O[C@H](C)O[C@@H]3CC[C@@H]21)C4=CC=CO4",
    "Barettin": "C1=CC=C(C=C1)CC(C(=O)N2CCC[C@H]2C(=O)NC(C(=O)O)CC3=CNC4=C3C=C(C=C4)Br)N",
    "Stypoldione": "CC1(C)CCCC2(C)C1CCC3=C2C=CC(=O)C3=O",
    "Lamellarin_H": "COC1=C(C=C2C(=C1)C3=C(C(=N2)C4=CC(=C(C=C4)O)OC)C5=CC(=C(C=C5O3)OC)O)O",
    "Gymnodimine": "C[C@@H]1CC[C@@]2(CC[C@@H]3C[C@H]4C(=C[C@H]5[C@@H]4C[C@@H]6[C@H]5CC=CC=C[C@@H]7[C@H]6C[C@@](O7)(CC=C3)O2)N1)C",
    "Bromophycolide_A": "CC1=C(C(=C(C(=C1)Br)O)Br)C2=C[C@@H]3[C@@H]([C@H](C2)C)O[C@H](C3)C(C)C",
    "Bastadin_6": "C1=CC(=C(C=C1C[C@H](C(=O)O)N)OC2=C(C=C(C=C2)C[C@H](C(=O)O)N)O)OC3=CC(=CC(=C3O)Br)Br"
}

# Run the AI engine
run_phase5_ai_generation(controls_14)


